# 图像生成高级教程

本教程涵盖高级采样、图像编辑、条件控制、部署优化等主题。

## 目录
1. [高级采样技术](#1-高级采样技术)
2. [图像编辑](#2-图像编辑)
3. [条件控制](#3-条件控制)
4. [训练技巧](#4-训练技巧)
5. [部署优化](#5-部署优化)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

## 1. 高级采样技术

### 1.1 Euler 采样器

In [ ]:
class EulerSampler:
    """Euler 采样器 - 简单高效的 ODE 求解"""
    def __init__(self, num_steps=50):
        self.num_steps = num_steps
    
    def get_sigmas(self, num_steps):
        """生成噪声调度"""
        return torch.linspace(1, 0.01, num_steps + 1)
    
    @torch.no_grad()
    def sample(self, model, shape):
        x = torch.randn(shape).to(device)
        sigmas = self.get_sigmas(self.num_steps).to(device)
        
        for i in range(len(sigmas) - 1):
            sigma = sigmas[i]
            sigma_next = sigmas[i + 1]
            
            # 模拟去噪预测
            denoised = x - sigma * torch.randn_like(x) * 0.1
            d = (x - denoised) / sigma
            dt = sigma_next - sigma
            x = x + d * dt
        
        return x

# 测试
sampler = EulerSampler(num_steps=20)
sample = sampler.sample(None, (1, 4, 64, 64))
print(f'Euler 采样输出: {sample.shape}')

### 1.2 Euler Ancestral 采样器

In [ ]:
class EulerAncestralSampler:
    """Euler Ancestral - 添加随机性增加多样性"""
    def __init__(self, num_steps=50, eta=1.0):
        self.num_steps = num_steps
        self.eta = eta  # 0=确定性, 1=完全随机
    
    @torch.no_grad()
    def sample(self, model, shape):
        x = torch.randn(shape).to(device)
        sigmas = torch.linspace(1, 0.01, self.num_steps + 1).to(device)
        
        for i in range(len(sigmas) - 1):
            sigma, sigma_next = sigmas[i], sigmas[i + 1]
            
            denoised = x - sigma * torch.randn_like(x) * 0.1
            
            # 计算噪声量
            sigma_up = min(sigma_next, self.eta * (sigma_next**2 * (sigma**2 - sigma_next**2) / sigma**2)**0.5)
            sigma_down = (sigma_next**2 - sigma_up**2)**0.5
            
            d = (x - denoised) / sigma
            x = x + d * (sigma_down - sigma)
            
            if sigma_next > 0:
                x = x + torch.randn_like(x) * sigma_up
        
        return x

sampler_a = EulerAncestralSampler(num_steps=20, eta=0.5)
sample_a = sampler_a.sample(None, (1, 4, 64, 64))
print(f'Euler Ancestral 输出: {sample_a.shape}')

In [ ]:
# 可视化采样器对比
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
samplers = ['DDIM', 'Euler', 'Euler-A']
steps = [50, 20, 20]
quality = ['高', '高', '高']
diversity = ['低', '低', '高']

for ax, name, step, q, d in zip(axes, samplers, steps, quality, diversity):
    ax.text(0.5, 0.7, name, ha='center', fontsize=14, fontweight='bold')
    ax.text(0.5, 0.5, f'推荐步数: {step}', ha='center', fontsize=11)
    ax.text(0.5, 0.35, f'质量: {q} | 多样性: {d}', ha='center', fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.add_patch(plt.Rectangle((0.1, 0.1), 0.8, 0.8, fill=False, ec='black', lw=2))

plt.suptitle('采样器对比', fontsize=14)
plt.tight_layout()
plt.show()

## 2. 图像编辑

### 2.1 Inpainting (图像修复)

In [ ]:
class InpaintingPipeline:
    """图像修复管道"""
    def __init__(self, num_steps=50):
        self.num_steps = num_steps
    
    @torch.no_grad()
    def inpaint(self, image, mask, noise_level=0.5):
        """
        Args:
            image: 原始图像 [B, C, H, W]
            mask: 修复区域 (1=修复, 0=保留) [B, 1, H, W]
        """
        # 添加噪声
        noise = torch.randn_like(image)
        noisy = image * (1 - noise_level) + noise * noise_level
        
        # 模拟去噪过程
        for t in range(self.num_steps):
            alpha = 1 - t / self.num_steps
            denoised = noisy - noise * 0.02
            
            # 混合：保留原始区域
            noisy = denoised * mask + image * (1 - mask)
        
        return noisy

# 测试
inpaint = InpaintingPipeline()
image = torch.randn(1, 3, 256, 256).to(device)
mask = torch.zeros(1, 1, 256, 256).to(device)
mask[:, :, 64:192, 64:192] = 1  # 中心区域修复

result = inpaint.inpaint(image, mask)
print(f'Inpainting 输出: {result.shape}')

### 2.2 Image-to-Image

In [ ]:
class Img2ImgPipeline:
    """图像到图像变换"""
    def __init__(self, num_steps=50):
        self.num_steps = num_steps
    
    @torch.no_grad()
    def transform(self, image, strength=0.75):
        """
        Args:
            strength: 变换强度 (0=不变, 1=完全重新生成)
        """
        start_step = int(self.num_steps * (1 - strength))
        
        # 添加噪声到起始步
        noise = torch.randn_like(image)
        noisy = image * (1 - strength) + noise * strength
        
        # 从中间开始去噪
        for t in range(start_step, self.num_steps):
            alpha = t / self.num_steps
            noisy = noisy - noise * 0.02 * alpha
        
        return noisy

# 测试不同强度
img2img = Img2ImgPipeline()
image = torch.randn(1, 3, 256, 256).to(device)

for strength in [0.3, 0.5, 0.75]:
    result = img2img.transform(image, strength)
    print(f'Strength {strength}: 输出形状 {result.shape}')

## 3. 条件控制

### 3.1 多 ControlNet 组合

In [ ]:
class MultiControlNet(nn.Module):
    """组合多个 ControlNet"""
    def __init__(self, num_controls=3, hidden_dim=320):
        super().__init__()
        self.controls = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(3, hidden_dim, 3, padding=1),
                nn.SiLU(),
                nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
            ) for _ in range(num_controls)
        ])
        self.weights = nn.Parameter(torch.ones(num_controls))
    
    def forward(self, conditions):
        """conditions: list of [B, 3, H, W]"""
        outputs = []
        for ctrl, cond, w in zip(self.controls, conditions, self.weights):
            outputs.append(ctrl(cond) * w)
        return sum(outputs)

# 测试
multi_ctrl = MultiControlNet().to(device)
canny = torch.randn(1, 3, 64, 64).to(device)
depth = torch.randn(1, 3, 64, 64).to(device)
pose = torch.randn(1, 3, 64, 64).to(device)

output = multi_ctrl([canny, depth, pose])
print(f'Multi-ControlNet 输出: {output.shape}')

### 3.2 Classifier-Free Guidance

In [ ]:
def classifier_free_guidance(noise_pred_cond, noise_pred_uncond, guidance_scale=7.5):
    """
    CFG: 增强条件生成效果
    
    公式: pred = uncond + scale * (cond - uncond)
    """
    return noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

# 可视化 CFG 效果
scales = [1.0, 3.0, 7.5, 15.0, 20.0]
effects = ['无引导', '轻微', '推荐', '强烈', '过饱和']

fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(range(len(scales)), scales, color='steelblue')
ax.set_yticks(range(len(scales)))
ax.set_yticklabels([f'{s} ({e})' for s, e in zip(scales, effects)])
ax.set_xlabel('Guidance Scale')
ax.set_title('Classifier-Free Guidance 强度效果')
ax.axvline(x=7.5, color='red', linestyle='--', label='推荐值')
plt.tight_layout()
plt.show()

## 4. 训练技巧

### 4.1 Noise Offset

In [ ]:
def add_noise_with_offset(images, noise, offset_strength=0.1):
    """
    Noise Offset: 改善暗色/亮色图像生成
    """
    # 添加全局噪声偏移
    offset = torch.randn(images.shape[0], 1, 1, 1, device=images.device)
    noise_with_offset = noise + offset_strength * offset
    return noise_with_offset

# 测试
images = torch.randn(4, 3, 64, 64).to(device)
noise = torch.randn_like(images)
noise_offset = add_noise_with_offset(images, noise)
print(f'原始噪声均值: {noise.mean():.4f}')
print(f'偏移噪声均值: {noise_offset.mean():.4f}')

### 4.2 SNR 加权损失

In [ ]:
def snr_weighted_loss(noise_pred, noise, timesteps, gamma=5.0):
    """
    SNR 加权: 平衡不同噪声水平的损失贡献
    """
    # 模拟 SNR 计算
    snr = 1.0 / (timesteps.float() / 1000 + 0.01)
    weight = torch.clamp(snr, max=gamma) / snr
    
    # 加权 MSE
    mse = F.mse_loss(noise_pred, noise, reduction='none').mean(dim=[1, 2, 3])
    return (weight * mse).mean()

# 测试
noise_pred = torch.randn(4, 4, 64, 64).to(device)
noise = torch.randn(4, 4, 64, 64).to(device)
timesteps = torch.randint(0, 1000, (4,)).to(device)

loss = snr_weighted_loss(noise_pred, noise, timesteps)
print(f'SNR 加权损失: {loss.item():.4f}')

## 5. 部署优化

### 5.1 模型量化

In [ ]:
def quantize_model_dynamic(model):
    """动态 INT8 量化"""
    return torch.quantization.quantize_dynamic(
        model, {nn.Linear, nn.Conv2d}, dtype=torch.qint8
    )

# 测试
model = nn.Sequential(
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 256)
)

original_size = sum(p.numel() * 4 for p in model.parameters())  # FP32
quantized = quantize_model_dynamic(model)

print(f'原始模型大小: {original_size / 1024:.1f} KB')
print(f'量化后约: {original_size / 4 / 1024:.1f} KB (理论 4x 压缩)')

### 5.2 批处理优化

In [ ]:
class BatchedGeneration:
    """批处理生成优化"""
    def __init__(self, batch_size=4):
        self.batch_size = batch_size
    
    def generate_batch(self, prompts, num_steps=20):
        """批量生成"""
        results = []
        for i in range(0, len(prompts), self.batch_size):
            batch = prompts[i:i+self.batch_size]
            # 模拟批量生成
            images = torch.randn(len(batch), 3, 512, 512)
            results.extend(images)
        return results

# 测试
generator = BatchedGeneration(batch_size=4)
prompts = ['prompt_' + str(i) for i in range(10)]
results = generator.generate_batch(prompts)
print(f'生成 {len(results)} 张图像')

## 总结

| 主题 | 关键技术 | 效果 |
|:-----|:---------|:-----|
| 采样 | Euler/DPM-Solver | 20步高质量 |
| 编辑 | Inpainting/Img2Img | 精确控制 |
| 控制 | Multi-ControlNet/CFG | 多条件组合 |
| 优化 | 量化/批处理 | 2-4x 加速 |